In [ ]:
# Cargamos todos los paquetes necesarios

import matplotlib.pyplot as plt
from matplotlib.pyplot import imread
import numpy as np
from scipy import signal
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
%matplotlib inline

# Redes Neuronales Convolucionales

Las redes neuronales convolucionales son una extensión de las redes neuronales que conocemos hasta el momento, son consideradas métodos de aprendizaje profundo o Deep Learning. Son generalmente usadas en procesamiento de imágenes como lo son:
- Análisis de imágenes
- Clasificación de imágenes
- Lenguaje Natural

La arquitectura de una red convolucional se basa en secuencia de capas de neuronas, alternando entre capas convolucionales y capas de agrupación. Una representación clásica de las redes convolucionales es la siguiente:

<img src="img/CNN.png"/>

# Convolución y capas de convolución

La convolución no es un concepto nuevo, esta técnica ha sido utilizada en procesamiento de señales e imágenes. En procesamiento de imágenes se utilizan filtros predefinidos para resaltar o disminuir ciertas características de las imágenes. Los filtros son aplicados en un barrido por toda la imagen para producir una imagen alterada.

<img src="img/same_padding_no_strides.gif"/>
<h6>Vincent Dumoulin, Francesco Visin - A guide to convolution arithmetic for deep learning</h6>

<img src="img/conv.gif" width=50%/>   

## Filtros

In [ ]:
image = imread("data/arbol.png")
plt.figure(figsize=(6,6))
plt.imshow(image)
plt.axis('off')
image.shape

In [ ]:
# Función  para convertir una imagen a escala de grises
def to_greyscale(a):
    #0.299 R + 0.587 G + 0.114 B
    return 0.299*a[0] + 0.587*a[1] + 0.114*a[2]

bw_image = np.apply_along_axis(to_greyscale, 2, image)
plt.figure(figsize=(6,6))
plt.imshow(bw_image,cmap=plt.cm.gray)
plt.axis('off')
bw_image.shape

In [ ]:
smooth_filter = np.full((3, 3), 1/9)
smooth_filter

In [ ]:
new_image = signal.convolve2d(bw_image, smooth_filter, mode='same')
plt.figure(figsize=(6,6))
plt.imshow(new_image,cmap=plt.cm.gray)
plt.axis('off')
bw_image.shape

In [ ]:
kernel_filter = np.array([[-1/9,-1/9,-1/9],
                          [-1/9, 17/9,-1/9],
                          [-1/9,-1/9,-1/9]])
new_image = signal.convolve2d(bw_image, kernel_filter, mode='same')
plt.figure(figsize=(6,6))
plt.axis('off')
plt.imshow(new_image,cmap=plt.cm.gray);

In [ ]:
kernel_filter = np.array([[-1,-2,-1],
                          [ 0, 0, 0],
                          [ 1, 2, 1]])
new_image1 = signal.convolve2d(bw_image, kernel_filter, mode='same')

kernel_filter = np.array([[-1, 0, 1],
                          [-2, 0, 2],
                          [-1, 0, 1]])
new_image2 = signal.convolve2d(bw_image, kernel_filter, mode='same')


plt.figure(figsize=(12,12))
plt.subplot(221)
plt.axis('off')
plt.title("Horizontal")
plt.imshow(new_image1,cmap=plt.cm.gray)
plt.subplot(222)
plt.axis('off')
plt.title("Vertical")
plt.imshow(new_image2,cmap=plt.cm.gray);

¿Cuáles son las características importantes por resaltar para la clasificación de las imágenes?

## Capa de convolución

En las redes de convolución los filtros utilizados no están definidos. Los filtros son "aprendidos" en tiempo de entrenamiento los cuales resaltan las características más importantes de la imagen. Como las redes construyen los filtros directamente a partir de la función de error, los filtros son más efectivos que cualquier filtro hecho manualmente por un humano.

De hecho, la mayoría de las veces las redes de convolución generan filtros que capturan conceptos abstractos como por ejemplo el borde del mentón o la silueta de la oreja entre otras singularidades. Si agregamos más capas los filtros aprenden a capturar cosas más complejas como por ejemplo ojos o boca.
El objetivo de la convolución es extraer las características de alto nivel desde una imagen de entrada.

Algunos parámetros que podemos especificar son:
filters: Determina el número de filtros generados en la capa.
padding: Indica la forma que tendrá la imagen resultante después del barrido.
kernel_size: Determina el tamaño de los filtros creados.
strides = Determina el número de pixeles que debe ignorar entre movimientos del barrido.

Puedes leer más sobre los parámetros en la documentación oficial:
https://keras.io/layers/convolutional/

## Capa de agrupación

La capa de agrupación funciona de una manera similar a la capa convolución donde se hace un recorrido a la imagen aplicando una función a cada sección de la imagen. El objetivo de esta capa es agrupar en una imagen las características más relevantes. Esto se realiza con 2 posibles funciones. La primera de ellas obtener el promedio de la sección analizada, la segunda es obtener el valor máximo de la sección analizada.

Documentación oficial: https://keras.io/layers/pooling/

<img src="img/no_padding_strides.gif"/>

<img src="img/pooling.png" width="50%"/>

# Lectura de los datos

In [ ]:
df = pd.read_csv('data/digit/train.csv')

In [ ]:
print("El conjunto de datos tiene {} filas y {} columnas".format(df.shape[0],df.shape[1]))
df.head()

In [ ]:
y = df.label.values
x = df.drop("label",axis=1)

In [ ]:
x = x.values
x = x / 255.0

In [ ]:
x = x.reshape((-1,28,28,1))

In [ ]:
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    number = x[i].reshape(28,28)
    plt.imshow(number, cmap=plt.cm.binary)
    plt.xlabel(y[i])
plt.show()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.30, random_state=103)

In [ ]:
x_train.shape

# Construyendo y entrenando la NN

<img src="img/CNN_detailed.png"/>

In [ ]:
from tensorflow import keras
from tensorflow.keras import datasets, layers, models

input_layer = x_train

model = models.Sequential()

# Creando Conv2D_1
model.add(
    layers.Conv2D(filters = 32, 
                  kernel_size = (5, 5),
                  activation='relu',
                  padding="same",
                  input_shape=(28,28,1,))
)

# Creando Pooling2D_1
model.add(
    layers.MaxPooling2D(pool_size = (2, 2),
                        strides=2)
)

# Creando Conv2D_2
model.add(
    layers.Conv2D(filters = 64, 
                  kernel_size = (5, 5),
                  activation='relu',
                  padding="same",
                  input_shape=input_layer.shape)
)

# Creando Pooling2D_2
model.add(
    layers.MaxPooling2D(pool_size = (2, 2),
                        strides=2)
)

model.summary()

In [ ]:
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dropout(rate = 0.4))
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dropout(rate = 0.4))
model.add(layers.Dense(10, activation='softmax'))
model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(x_train, y_train, 
          validation_data = (x_test, y_test),
          epochs=10,
          batch_size = 100);

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)

print("Test acc:", test_acc)

# Probando la NN

In [ ]:
predictions = model.predict(x_test)
y_hat = np.argmax(predictions,axis=1)

In [ ]:
errores = x_test[y_test != y_hat]
print("Elementos de prueba: {}".format(y_test.shape[0]))
errores_count = errores.shape[0]
errores_count
print("Errores identificados: {}".format(errores_count))
porcentaje_error = errores_count *100 / y_test.shape[0]
print("Porcentaje de error: {} %".format(porcentaje_error))

In [ ]:
real_labels = y_test[y_test != y_hat]
predicted_labels = y_hat[y_test != y_hat]

for j in range(round(errores.shape[0]/5)):
    plt.figure(figsize=(10,10))
    for i in range(min(5,errores_count-5*j)):
        plt.subplot(1,min(5,errores_count-5*j),i+1)
        plt.xticks([])
        plt.yticks([])
        plt.grid(False)
        number = errores[5*j+i].reshape(28,28)
        plt.imshow(number, cmap=plt.cm.binary)
        plt.xlabel("Real: {} Prediccion: {}".format
                   (real_labels[5*j+i],predicted_labels[5*j+i]))
    plt.show()

# Matriz de confusión

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

mod_confusion_matrix = confusion_matrix(y_test, y_hat)
for i in range(10):
    mod_confusion_matrix[i][i] = 0
    
    
fig = plt.figure(figsize=(12,10))
ax = fig.add_subplot(1,1,1) 
sns.heatmap(mod_confusion_matrix, linewidth=0.5, annot=True, cmap="YlGnBu")
plt.ylabel("Etiquetas Reales")
plt.xlabel("Etiquetas Predecidas")
ax.set_ylim(10.5,-0.5);